# HAM10000 Multimodal Conformal Prediction

This notebook applies the shared conformal classification classes to the HAM10000 multimodal Random Forest experiment.

The multimodal classifier was built from:

- image features extracted with a pretrained PyTorch ResNet18 encoder
- encoded HAM10000 metadata features
- feature-level fusion by concatenation
- Random Forest classification on the fused features

Here we use the validation split as the calibration set and the test split to evaluate conformal prediction sets.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path('/Users/nikos/Documents/Codex/2026-07-25/s/outputs/ham10000-pytorch')
SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from conformal_prediction import ClassificationScores, SplitConformalClassifier, evaluate_prediction_sets

In [ ]:
RF_RUN_DIR = Path('/Users/nikos/Documents/Codex/2026-07-25/s/outputs/ham10000-random-forest')

FEATURES_NPZ = RF_RUN_DIR / 'fused_features.npz'
MODEL_JOBLIB = RF_RUN_DIR / 'random_forest.joblib'
SPLITS_CSV = RF_RUN_DIR / 'splits.csv'

ALPHA = 0.10
CONFIDENCE_LEVEL = 1.0 - ALPHA
SCORE_TYPES = ['probability', 'cumulative', 'high_probability']

NOTEBOOK_OUTPUT_DIR = RF_RUN_DIR / 'notebook_conformal_alpha_0_10'
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = {
    'akiec': "Actinic keratosis / Bowen's disease",
    'bcc': 'Basal cell carcinoma',
    'bkl': 'Benign keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevus',
    'vasc': 'Vascular lesion',
}
label_names = sorted(LABEL_NAMES)
target_names = [f'{label}: {LABEL_NAMES[label]}' for label in label_names]

FEATURES_NPZ, MODEL_JOBLIB, SPLITS_CSV, NOTEBOOK_OUTPUT_DIR

## 2. Load fused multimodal features

`fused_features.npz` contains the already-created multimodal vectors:

- `train_features`: fused image + metadata features for training
- `val_features`: calibration features
- `test_features`: test features
- matching label arrays for each split

In [ ]:
with np.load(FEATURES_NPZ, allow_pickle=False) as data:
    arrays = {name: data[name] for name in data.files}

feature_shapes = pd.DataFrame(
    [
        {'array': name, 'shape': str(value.shape), 'dtype': str(value.dtype)}
        for name, value in arrays.items()
    ]
)
feature_shapes

## 3. Load the trained multimodal Random Forest

The Random Forest was trained on the fused multimodal representation. The conformal method only needs a classifier with `predict_proba`, because conformal classification turns class probabilities into prediction sets.

In [ ]:
payload = joblib.load(MODEL_JOBLIB)
classifier = payload['classifier'] if isinstance(payload, dict) else payload

print(type(classifier))
print('Model classes:', classifier.classes_)

In [ ]:
test_point_predictions = classifier.predict(arrays['test_features']).astype(int)

point_metrics = {
    'test_accuracy': accuracy_score(arrays['test_labels'], test_point_predictions),
    'test_balanced_accuracy': balanced_accuracy_score(arrays['test_labels'], test_point_predictions),
}
point_metrics

## 4. Load test image metadata

This is used only to make the prediction CSV easier to read. The conformal calculation itself uses the fused feature arrays and labels.

In [ ]:
splits = pd.read_csv(SPLITS_CSV)
test_rows = splits[splits['split'] == 'test'].reset_index(drop=True)

print(test_rows.shape)
test_rows.head()

## 5. Calibrate conformal predictors

For each score function, we:

1. wrap the trained Random Forest with `SplitConformalClassifier(prefit=True)`
2. calibrate on the validation split
3. build prediction sets on the test split
4. compute coverage and prediction-set size

In [ ]:
def labels_from_set(prediction_set):
    return ';'.join(label_names[int(index)] for index in prediction_set)


def make_predictions_frame(probabilities, predicted, prediction_sets, set_metrics):
    if len(test_rows) == len(arrays['test_labels']):
        frame = test_rows[['image_id', 'lesion_id', 'dx']].copy()
    else:
        frame = pd.DataFrame({'row_index': np.arange(len(arrays['test_labels']))})

    y_true = arrays['test_labels']
    frame['true_index'] = y_true
    frame['true_label'] = [label_names[int(index)] for index in y_true]
    frame['predicted_index'] = predicted
    frame['predicted_label'] = [label_names[int(index)] for index in predicted]
    frame['covered'] = set_metrics['covered']
    frame['prediction_set_size'] = set_metrics['set_sizes']
    frame['prediction_set'] = [labels_from_set(pred_set) for pred_set in prediction_sets]
    frame['predicted_probability'] = probabilities[np.arange(len(predicted)), predicted]
    frame['true_probability'] = probabilities[np.arange(len(y_true)), y_true]

    for index, label in enumerate(label_names):
        frame[f'probability_{label}'] = probabilities[:, index]

    return frame


def make_class_coverage_frame(set_metrics):
    rows = []
    for class_index, values in set_metrics['per_class'].items():
        rows.append(
            {
                'class_index': int(class_index),
                'class_label': label_names[int(class_index)],
                'support': values['support'],
                'coverage': values['coverage'],
                'average_set_size': values['average_set_size'],
            }
        )
    return pd.DataFrame(rows)


def run_conformal_score(score_type):
    cp = SplitConformalClassifier(
        model=classifier,
        alpha=ALPHA,
        score=ClassificationScores(score_type=score_type),
        prefit=True,
    )
    cp.calibrate(arrays['val_features'], arrays['val_labels'])

    probabilities = cp.predict_proba(arrays['test_features'])
    predicted = cp.predict(arrays['test_features']).astype(int)
    prediction_sets = cp.predict_set(arrays['test_features'])
    set_metrics = evaluate_prediction_sets(arrays['test_labels'], prediction_sets, cp.classes_)

    predictions = make_predictions_frame(probabilities, predicted, prediction_sets, set_metrics)
    class_coverage = make_class_coverage_frame(set_metrics)

    summary_row = {
        'score_type': score_type,
        'alpha': ALPHA,
        'confidence_level': CONFIDENCE_LEVEL,
        'q_hat': cp.q_hat_,
        'calibration_size': len(arrays['val_labels']),
        'test_size': len(arrays['test_labels']),
        'point_accuracy': point_metrics['test_accuracy'],
        'point_balanced_accuracy': point_metrics['test_balanced_accuracy'],
        'coverage': set_metrics['coverage'],
        'average_set_size': set_metrics['average_set_size'],
        'median_set_size': set_metrics['median_set_size'],
        'singleton_fraction': set_metrics['singleton_fraction'],
        'max_set_size': set_metrics['max_set_size'],
    }

    return {
        'cp': cp,
        'summary_row': summary_row,
        'predictions': predictions,
        'class_coverage': class_coverage,
        'calibration_scores': pd.DataFrame({'calibration_score': cp.calibration_scores_}),
    }

In [ ]:
results = {score_type: run_conformal_score(score_type) for score_type in SCORE_TYPES}
summary = pd.DataFrame([result['summary_row'] for result in results.values()])

summary

## 6. Per-class conformal coverage

HAM10000 is imbalanced, so the overall coverage can look reasonable while rare classes still have unstable behavior. The next table shows the class-level coverage for the probability score.

In [ ]:
probability_class_coverage = results['probability']['class_coverage']
probability_class_coverage

## 7. Inspect individual conformal prediction sets

`prediction_set` contains all labels included by conformal prediction. A singleton set behaves like an ordinary point prediction; a larger set means the model is uncertain among several plausible labels.

In [ ]:
probability_predictions = results['probability']['predictions']
columns_to_view = [
    'image_id',
    'dx',
    'predicted_label',
    'covered',
    'prediction_set_size',
    'prediction_set',
    'predicted_probability',
    'true_probability',
]
probability_predictions[columns_to_view].head(15)

## 8. Save notebook outputs

In [ ]:
summary.to_csv(NOTEBOOK_OUTPUT_DIR / 'conformal_summary.csv', index=False)

for score_type, result in results.items():
    result['predictions'].to_csv(NOTEBOOK_OUTPUT_DIR / f'{score_type}_test_predictions.csv', index=False)
    result['class_coverage'].to_csv(NOTEBOOK_OUTPUT_DIR / f'{score_type}_coverage_by_class.csv', index=False)
    result['calibration_scores'].to_csv(NOTEBOOK_OUTPUT_DIR / f'{score_type}_calibration_scores.csv', index=False)

print(f'Saved notebook outputs to: {NOTEBOOK_OUTPUT_DIR}')

## 9. Interpretation

For `alpha = 0.10`, the nominal conformal confidence level is 90%. The `probability` and `high_probability` scores usually produce smaller prediction sets. The `cumulative` score is more conservative here: it gives higher empirical coverage, but larger prediction sets.

Because HAM10000 is strongly imbalanced, the per-class table should always be read together with the overall coverage.